# 第二プロジェクト DB 半年前差分

新しい `db.zst.part0` + `db.zst.part1` と、`https://sr.ht/~kasosuta/dai2db/` にある旧DBを比較します。

出力:
- `summary.json`
- `added.csv`
- `removed.csv`
- `changed.csv`
- 上記をまとめた `db_diff.zip`

比較はコメントID基準です。旧DBと新DBで列名が多少違っても主要列を自動検出します。


In [ ]:
!apt-get -qq update
!apt-get -qq install -y zstd
!pip -q install beautifulsoup4 requests pandas


In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import csv
import gzip
import json
import os
import shutil
import sqlite3
import subprocess
import zipfile

import pandas as pd
import requests
from bs4 import BeautifulSoup

NEW_BASE = "https://raw.githubusercontent.com/22552/kasotest/main/"
OLD_PAGE = "https://sr.ht/~kasosuta/dai2db/"
WORK = Path("/content/dai2db-diff")
WORK.mkdir(exist_ok=True)

session = requests.Session()
session.headers["User-Agent"] = "dai2db-colab-diff/1"


In [ ]:
def download(url, dst):
    dst = Path(dst)
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length") or 0)
        done = 0
        with dst.open("wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r{dst.name}: {done / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MiB", end="")
    print()
    return dst

def unpack_db(src, dst):
    src, dst = Path(src), Path(dst)
    name = src.name.lower()

    if name.endswith(".zst"):
        subprocess.run(["zstd", "-d", "-f", str(src), "-o", str(dst)], check=True)
        return dst

    if name.endswith(".gz"):
        with gzip.open(src, "rb") as i, dst.open("wb") as o:
            shutil.copyfileobj(i, o)
        return dst

    if name.endswith(".zip"):
        with zipfile.ZipFile(src) as z:
            files = [x for x in z.namelist() if not x.endswith("/")]
            best = max(files, key=lambda x: (
                Path(x).suffix.lower() in {".sqlite", ".sqlite3", ".db"},
                z.getinfo(x).file_size,
            ))
            with z.open(best) as i, dst.open("wb") as o:
                shutil.copyfileobj(i, o)
        return dst

    shutil.copyfile(src, dst)
    return dst

def is_sqlite(path):
    with open(path, "rb") as f:
        return f.read(16) == b"SQLite format 3\x00"


In [ ]:
# 新DB: GitHub上の2分割ファイルを結合して展開
p0 = download(urljoin(NEW_BASE, "db.zst.part0"), WORK / "db.zst.part0")
p1 = download(urljoin(NEW_BASE, "db.zst.part1"), WORK / "db.zst.part1")

new_zst = WORK / "new.db.zst"
with new_zst.open("wb") as out:
    for p in (p0, p1):
        with p.open("rb") as f:
            shutil.copyfileobj(f, out)

new_db = unpack_db(new_zst, WORK / "new.sqlite")
assert is_sqlite(new_db), "新DBがSQLiteとして読めません"
print("new:", new_db, f"{new_db.stat().st_size / 1024 / 1024:.1f} MiB")


In [ ]:
# 旧DB: sr.ht のページからDB候補を自動検出
r = session.get(OLD_PAGE, timeout=60)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

exts = (".zst", ".sqlite", ".sqlite3", ".db", ".gz", ".zip")
urls = []
for a in soup.find_all("a", href=True):
    u = urljoin(OLD_PAGE, a["href"])
    path = urlparse(u).path.lower()
    if path.endswith(exts):
        urls.append(u)

urls = list(dict.fromkeys(urls))
if not urls:
    raise RuntimeError(
        "旧DBへのリンクを自動検出できませんでした。"
        "このセルの OLD_DB_URL に実ファイルURLを直接指定してください。"
    )

def remote_size(u):
    try:
        h = session.head(u, allow_redirects=True, timeout=30)
        return int(h.headers.get("content-length") or 0)
    except Exception:
        return 0

ranked = sorted(
    urls,
    key=lambda u: (
        "db" in Path(urlparse(u).path).name.lower() or "sqlite" in u.lower(),
        remote_size(u),
    ),
    reverse=True,
)

print("旧DB候補:")
for i, u in enumerate(ranked):
    print(i, u)

OLD_DB_URL = ranked[0]
print("使用:", OLD_DB_URL)

old_archive = download(OLD_DB_URL, WORK / Path(urlparse(OLD_DB_URL).path).name)
old_db = unpack_db(old_archive, WORK / "old.sqlite")
assert is_sqlite(old_db), "旧DBがSQLiteとして読めません"
print("old:", old_db, f"{old_db.stat().st_size / 1024 / 1024:.1f} MiB")


In [ ]:
ALIASES = {
    "id": ("id", "comment_id"),
    "parent_id": ("parent_id", "parent", "reply_to", "reply_to_id"),
    "is_reply": ("is_reply",),
    "user": ("user", "username", "author", "author_name"),
    "user_id": ("user_id", "author_id"),
    "datetime": ("datetime", "datetime_created", "created_at", "timestamp", "date"),
    "content": ("content", "text", "body"),
}

def qident(s):
    return '"' + s.replace('"', '""') + '"'

def detect_table(db_path):
    db = sqlite3.connect(db_path)
    best = None
    try:
        tables = [
            x[0] for x in db.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
            )
        ]
        for table in tables:
            cols = [x[1] for x in db.execute(f"PRAGMA table_info({qident(table)})")]
            lower = {c.lower(): c for c in cols}
            mapping = {}
            for logical, aliases in ALIASES.items():
                for alias in aliases:
                    if alias in lower:
                        mapping[logical] = lower[alias]
                        break
            score = (100 if table.lower() == "comments" else 0) + len(mapping) * 10
            if "id" not in mapping:
                continue
            if best is None or score > best[0]:
                best = (score, table, mapping, cols)
    finally:
        db.close()

    if best is None:
        raise RuntimeError(f"コメントIDらしい列を持つテーブルが見つかりません: {db_path}")
    return best[1], best[2], best[3]

new_table, new_map, new_cols = detect_table(new_db)
old_table, old_map, old_cols = detect_table(old_db)

print("new table:", new_table, new_map)
print("old table:", old_table, old_map)


In [ ]:
def norm_select(schema, table, mapping):
    out = []
    for logical in ALIASES:
        if logical in mapping:
            expr = f"{qident(mapping[logical])}"
            if logical in {"id", "parent_id", "is_reply", "user_id"}:
                expr = f"CAST({expr} AS INTEGER)"
            else:
                expr = f"CAST({expr} AS TEXT)"
            out.append(f"{expr} AS {qident(logical)}")
        else:
            out.append(f"NULL AS {qident(logical)}")
    return f"SELECT {', '.join(out)} FROM {schema}.{qident(table)}"

db = sqlite3.connect(":memory:")
db.execute("ATTACH DATABASE ? AS newdb", (str(new_db),))
db.execute("ATTACH DATABASE ? AS olddb", (str(old_db),))
db.execute(f"CREATE TEMP VIEW new_norm AS {norm_select('newdb', new_table, new_map)}")
db.execute(f"CREATE TEMP VIEW old_norm AS {norm_select('olddb', old_table, old_map)}")

shared = [x for x in ALIASES if x != "id" and x in new_map and x in old_map]
print("比較フィールド:", shared)


In [ ]:
new_count = db.execute("SELECT COUNT(*) FROM new_norm").fetchone()[0]
old_count = db.execute("SELECT COUNT(*) FROM old_norm").fetchone()[0]
overlap = db.execute("""
    SELECT COUNT(*)
    FROM new_norm n
    JOIN old_norm o USING(id)
""").fetchone()[0]
added = db.execute("""
    SELECT COUNT(*)
    FROM new_norm n
    LEFT JOIN old_norm o USING(id)
    WHERE o.id IS NULL
""").fetchone()[0]
removed = db.execute("""
    SELECT COUNT(*)
    FROM old_norm o
    LEFT JOIN new_norm n USING(id)
    WHERE n.id IS NULL
""").fetchone()[0]

if shared:
    changed_where = " OR ".join(f"n.{qident(c)} IS NOT o.{qident(c)}" for c in shared)
    changed = db.execute(f"""
        SELECT COUNT(*)
        FROM new_norm n
        JOIN old_norm o USING(id)
        WHERE {changed_where}
    """).fetchone()[0]
else:
    changed_where = "0"
    changed = 0

field_changes = {}
for c in shared:
    field_changes[c] = db.execute(f"""
        SELECT COUNT(*)
        FROM new_norm n
        JOIN old_norm o USING(id)
        WHERE n.{qident(c)} IS NOT o.{qident(c)}
    """).fetchone()[0]

summary = {
    "old_rows": old_count,
    "new_rows": new_count,
    "delta_rows": new_count - old_count,
    "overlap_ids": overlap,
    "added_ids": added,
    "removed_ids": removed,
    "changed_common_ids": changed,
    "changed_by_field": field_changes,
    "old_source": OLD_DB_URL,
    "new_source": [
        urljoin(NEW_BASE, "db.zst.part0"),
        urljoin(NEW_BASE, "db.zst.part1"),
    ],
}

print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
print("追加サンプル")
display(pd.read_sql_query("""
    SELECT n.*
    FROM new_norm n
    LEFT JOIN old_norm o USING(id)
    WHERE o.id IS NULL
    ORDER BY n.id DESC
    LIMIT 30
""", db))

print("消滅サンプル")
display(pd.read_sql_query("""
    SELECT o.*
    FROM old_norm o
    LEFT JOIN new_norm n USING(id)
    WHERE n.id IS NULL
    ORDER BY o.id DESC
    LIMIT 30
""", db))

if changed:
    cols = ["n.id"] + sum(
        ([f"o.{qident(c)} AS old_{qident(c)}", f"n.{qident(c)} AS new_{qident(c)}"] for c in shared),
        [],
    )
    display(pd.read_sql_query(
        f"SELECT {', '.join(cols)} FROM new_norm n JOIN old_norm o USING(id) "
        f"WHERE {changed_where} ORDER BY n.id DESC LIMIT 30",
        db,
    ))


In [ ]:
def write_query_csv(path, sql):
    cur = db.execute(sql)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([d[0] for d in cur.description])
        while True:
            rows = cur.fetchmany(10000)
            if not rows:
                break
            w.writerows(rows)

write_query_csv(
    WORK / "added.csv",
    "SELECT n.* FROM new_norm n LEFT JOIN old_norm o USING(id) WHERE o.id IS NULL ORDER BY n.id"
)
write_query_csv(
    WORK / "removed.csv",
    "SELECT o.* FROM old_norm o LEFT JOIN new_norm n USING(id) WHERE n.id IS NULL ORDER BY o.id"
)

if shared:
    cols = ["n.id AS id"]
    for c in shared:
        cols += [
            f"o.{qident(c)} AS old_{qident(c)}",
            f"n.{qident(c)} AS new_{qident(c)}",
        ]
    write_query_csv(
        WORK / "changed.csv",
        f"SELECT {', '.join(cols)} FROM new_norm n JOIN old_norm o USING(id) "
        f"WHERE {changed_where} ORDER BY n.id"
    )
else:
    (WORK / "changed.csv").write_text("id\n", encoding="utf-8")

(WORK / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

bundle = shutil.make_archive("/content/db_diff", "zip", WORK, logger=None)
print(bundle)


In [ ]:
from google.colab import files
files.download("/content/db_diff.zip")
